# Express POST Routes

A POST route handles HTTP requests where the client **sends data to the server** — form submissions, JSON payloads, file uploads. Defined with `app.post()` or `router.post()`, and it needs body-parsing middleware to actually read the incoming data.

---

## Basic Setup

1. Import `express` and create an app instance.
2. Register body-parsing middleware (`express.json()` and/or `express.urlencoded()`).
3. Define the path and a handler callback taking `(req, res)`.
4. Start the server with `app.listen()`.

```javascript
const express = require('express');
const app = express();

// Middleware — must come BEFORE the routes that need it
app.use(express.json());                          // parses application/json
app.use(express.urlencoded({ extended: true }));  // parses HTML form posts

app.post('/api/data', (req, res) => {
  const incomingData = req.body;

  res.status(201).json({
    message: 'Data received successfully',
    data: incomingData
  });
});

app.listen(3000, () => {
  console.log('Server running on port 3000');
});
```

### ES Modules variant

If `"type": "module"` is set in `package.json`:

```javascript
import express from 'express';
const app = express();
```

---

## Body-Parsing Middleware

Express does **not** parse request bodies by default. Without middleware, `req.body` is `undefined`.

| Middleware | Parses | Content-Type |
|---|---|---|
| `express.json()` | JSON payloads | `application/json` |
| `express.urlencoded({ extended: true })` | HTML form data | `application/x-www-form-urlencoded` |
| `express.text()` | Raw text | `text/plain` |
| `express.raw()` | Buffer (webhooks, signatures) | `application/octet-stream` |
| `multer` (3rd party) | File uploads | `multipart/form-data` |

`extended: true` uses the `qs` library, which supports nested objects (`user[address][city]`). `false` uses Node's `querystring` and only handles flat key–value pairs.

### Body size limit

Default is `100kb`. Raise it explicitly when needed:

```javascript
app.use(express.json({ limit: '1mb' }));
```

Oversized payloads throw a `PayloadTooLargeError` (HTTP 413).

---

## Where the Data Lives

Three distinct sources — don't confuse them:

```javascript
app.post('/api/users/:teamId', (req, res) => {
  req.params.teamId;   // route parameter  → /api/users/42
  req.query.notify;    // query string     → ?notify=true
  req.body;            // request body     → the JSON/form payload
  req.headers;         // headers, e.g. authorization
});
```

POST data belongs in the body. Query params on a POST are legal but usually a smell.

---

## Validation

Never trust `req.body`. Validate before touching a database.

```javascript
app.post('/api/users', (req, res) => {
  const { name, email } = req.body;

  if (!name || !email) {
    return res.status(400).json({ error: 'name and email are required' });
  }

  // ... persist
  res.status(201).json({ id: 1, name, email });
});
```

Note the `return` — without it, execution continues and you risk `ERR_HTTP_HEADERS_SENT`.

For anything non-trivial, use a schema validator: **Zod**, **Joi**, or **express-validator**.

---

## Async Handlers & Error Handling

Most real POST handlers do I/O, so they're async.

```javascript
app.post('/api/orders', async (req, res, next) => {
  try {
    const order = await db.orders.create(req.body);
    res.status(201).json(order);
  } catch (err) {
    next(err);   // hand off to the error middleware
  }
});

// Error-handling middleware — 4 args, registered last
app.use((err, req, res, next) => {
  console.error(err);
  res.status(500).json({ error: 'Internal server error' });
});
```

> **Express 5 change:** rejected promises in async handlers are forwarded to the error handler automatically, so the `try/catch` + `next(err)` boilerplate is optional. In Express 4 it is required — an unhandled rejection there just hangs the request.

---

## Status Codes for POST

| Code | Meaning | When |
|---|---|---|
| `200 OK` | Success, no new resource | Action/RPC-style POST (e.g. `/login`, `/calculate`) |
| `201 Created` | New resource created | The standard for a successful create; pair with a `Location` header |
| `204 No Content` | Success, empty body | Fire-and-forget |
| `400 Bad Request` | Malformed input | Missing/invalid fields |
| `401 / 403` | Unauthenticated / forbidden | Auth failures |
| `409 Conflict` | Duplicate | Email already registered |
| `422 Unprocessable Entity` | Syntactically valid, semantically wrong | Failed schema validation |
| `500` | Server fault | Unexpected exception |

```javascript
res.status(201)
   .location(`/api/users/${user.id}`)
   .json(user);
```

---

## Modular Routing with `express.Router()`

Keeps large apps from collapsing into one file.

**`routes/users.js`**

```javascript
const express = require('express');
const router = express.Router();

// Path is relative to the mount point
router.post('/', (req, res) => {
  res.status(201).json({ created: req.body });
});

router.post('/:id/avatar', (req, res) => {
  res.json({ userId: req.params.id });
});

module.exports = router;
```

**`app.js`**

```javascript
const usersRouter = require('./routes/users');

app.use('/api/users', usersRouter);
// → POST /api/users
// → POST /api/users/:id/avatar
```

A `Router` is itself middleware, so it can carry its own middleware stack (auth, validation) independent of the rest of the app.

---

## POST vs PUT vs PATCH

| Verb | Purpose | Idempotent? |
|---|---|---|
| `POST` | Create a resource, or trigger an action | No — calling twice creates two records |
| `PUT` | Replace a resource wholesale at a known URI | Yes |
| `PATCH` | Partially update a resource | Not guaranteed |

Because POST isn't idempotent, retries and double-clicks can create duplicates. Guard with unique DB constraints or an idempotency key.

---

## Testing a POST Route

**curl**

```bash
curl -X POST http://localhost:3000/api/data \
  -H "Content-Type: application/json" \
  -d '{"name":"Harshit","role":"BI Engineer"}'
```

**fetch (browser / client)**

```javascript
const res = await fetch('/api/data', {
  method: 'POST',
  headers: { 'Content-Type': 'application/json' },
  body: JSON.stringify({ name: 'Harshit' })
});
const data = await res.json();
```

Or use Postman / Insomnia / Thunder Client (VS Code extension).

---

## Common Gotchas

- **`req.body` is `undefined`** → `express.json()` missing, or registered *after* the route.
- **Client didn't send `Content-Type: application/json`** → `express.json()` skips the body silently, leaving `{}`.
- **Middleware order matters.** `app.use()` calls run top-to-bottom; anything registered after a matching route never runs for it.
- **Forgetting `return` after `res.send()`** → double-response error.
- **Trailing-slash / path mismatch** → `/api/data` and `/api/data/` are distinct unless `strict routing` is off (it is by default).
- **CORS** — a browser POST from a different origin needs the `cors` middleware; a preflight `OPTIONS` request fires first.
- **`res.send()` vs `res.json()`** — `res.json()` sets the JSON content type and serialises; prefer it for API responses.

---

## Related

- [[Express Routing Basics]]
- [[Express Middleware]]
- [[HTTP Status Codes]]
- [[REST API Design]]

## References

- [Express routing guide](https://expressjs.com/en/5x/guide/routing/)
- [MDN — Express routes and controllers](https://developer.mozilla.org/en-US/docs/Learn_web_development/Extensions/Server-side/Express_Nodejs/routes)
- [GeeksforGeeks — `app.post()`](https://www.geeksforgeeks.org/web-tech/express-js-app-post-function/)
- [W3Schools — Node.js Express](https://www.w3schools.com/nodejs/nodejs_express.asp)